In [1]:
from pathlib import Path
from glob import glob
import pickle
import re

import pandas as pd
import numpy as np

In [2]:
from bikipy.behaviour.nort.trial import NortExperiment
from bikipy.utils.video import get_video_data

In [3]:
WORKING_DIR = Path("C:/Users/Can/Projects/Neuroscience/Imen/data")
NORT_DIR = WORKING_DIR / "nort"

HABIT_DIR = NORT_DIR / "habituation"
H_BEFORE_DIR = HABIT_DIR / "0_before"
H_AFTER_DIR = HABIT_DIR / "1_after"

NOVELTY_DIR = NORT_DIR / "novelty"
N_BEFORE_DIR = NOVELTY_DIR / "0_before_02.06.2020"
N_AFTER_DIR = NOVELTY_DIR / "1_after_24.08.2020"

ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "all.pickle")
BEFORE_ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "before.pickle")
AFTER_ANNOTATIONS_PATH = str(WORKING_DIR / "python_data" / "annotations" / "after.pickle")
IMPORTED_DLC_FILES = str(WORKING_DIR / "python_data" / "dlc.pickle")
RESULTS = str(WORKING_DIR / "python_data" / "results.pickle")

EXP_ID_FINDER = re.compile("\d+")
BORDER_DISTANCE = 3 * 224 / 40

with open(ANNOTATIONS_PATH, "rb") as infile:
    a1_1, a1_2, a1_3, a1_4, a2_1, a2_2, a2_3, a2_4, b1_1, b1_2, b1_3, b1_4, b2_1, b2_2, b2_3, b2_4 = pickle.load(infile)

In [4]:
app_to_obj = { "before": {
        1: {1: b1_1, 2: b1_2, 3: b1_3, 4: b1_4},
        2: {1: b2_1, 2: b2_2, 3: b2_3, 4: b2_4}
    },
    "after": {
        1: {1: a1_1, 2: a1_2, 3: a1_3, 4: a1_4},
        2: {1: a2_1, 2: a2_2, 3: a2_3, 4: a2_4}
    }
}

In [5]:
exp_info_df_0 = pd.read_excel(str(WORKING_DIR / "NORT_Round1.ods"), sheet_name=0)
exp_info_df_1 = pd.read_excel(str(WORKING_DIR / "NORT_Round1.ods"), sheet_name=1)

In [6]:
def get_animal_id_vs_exp_ids(info_df):
    exp_ids = np.array([int(EXP_ID_FINDER.findall(info)[-1]) for info in info_df["Video_file_name"]])
    animal_id = np.array(info_df["Animal"])

    result = {}
    for i in range(int(animal_id.min()), int(animal_id.max() + 1)):
        loc = np.where(animal_id == i)[0][:2]
        result[i] = tuple(exp_ids[loc])

    return result


def get_exp_id_vs_animal_id(id_exp):
    result = {}
    for animal, exps in id_exp.items():
        for exp in exps:
            result[exp] = animal
    return result

In [7]:
exp_animal = {
    "before": get_exp_id_vs_animal_id(get_animal_id_vs_exp_ids(exp_info_df_0)),
    "after": get_exp_id_vs_animal_id(get_animal_id_vs_exp_ids(exp_info_df_1))
}

In [8]:
def get_animal_id_vs_apparatus(info_df):
    animal_id = np.unique(info_df["Animal"])
    apparatus = info_df["Apparatus"]

    return {int(i): int(EXP_ID_FINDER.findall(app)[0]) for i, app in zip(animal_id, apparatus)}

In [9]:
id_app = {
    "before": get_animal_id_vs_apparatus(exp_info_df_0),
    "after": get_animal_id_vs_apparatus(exp_info_df_1)
}


In [10]:
def get_exp_id_vs_stage(exp_info_df):
    result = {}
    for row in exp_info_df[['Video_file_name', 'Stage']].iterrows():
        exp_idx = int(EXP_ID_FINDER.findall(Path(row[1][0]).stem)[0])
        stage = Path(row[1][1]).stem[-1:]
        result[exp_idx] = stage

    return result

In [11]:
exp_id_vs_stage = {
    "before": get_exp_id_vs_stage(exp_info_df_0),
    "after": get_exp_id_vs_stage(exp_info_df_1)
}

In [12]:
exp_ids_range_vs_exp_meta = {"before": {}, "after": {}}
exp_id_vs_coordinate_data_path = {"before": {}, "after": {}}
for time, paths in zip(exp_ids_range_vs_exp_meta, ((H_BEFORE_DIR, N_BEFORE_DIR), (H_AFTER_DIR, N_AFTER_DIR))):
    for stage, root in zip(("habituation", "novelty_observation"), paths):
        for data_path in glob(str(root / "*.h5")):
            exp_id = int(EXP_ID_FINDER.findall(Path(data_path).stem)[0])

            exp_id_vs_coordinate_data_path[time][exp_id] = data_path

        for data_path in glob(str(root / "*.mp4")):
            exp_id = int(EXP_ID_FINDER.findall(Path(data_path).stem)[0])

            exp_ids_range_vs_exp_meta[time][exp_id] = {
                "stage": stage,
                "recording_resolution": get_video_data(data_path)[1:]
            }

            if stage == "novelty_observation":
                animal_id = exp_animal[time][exp_id]
                apparatus = id_app[time][animal_id]

                stage = int(exp_id_vs_stage[time][exp_id])
                novelty_objs = app_to_obj[time][stage][apparatus]

                exp_ids_range_vs_exp_meta[time][exp_id] = {
                    **exp_ids_range_vs_exp_meta[time][exp_id],
                    **novelty_objs
                }


In [13]:
result = {}
for time in exp_ids_range_vs_exp_meta:
    result[time] = NortExperiment(
        exp_ids_range_vs_exp_meta[time],
        "nose",
        "mid-left_ear-right_ear",
        "mid-mid-left_ear-right_ear-tail",
        40,
        20,
        1 / 4 * np.pi,
        exp_id_vs_coordinate_data_path[time],
        14,
        midpoint_groups=[("left_ear", "right_ear"), ("mid-left_ear-right_ear", "tail")]
    )

TypeError: 'Future' object is not subscriptable

In [ ]:
# exp_ids_range_vs_exp_meta: Dict,
# nose_label: AnyStr,
# eye_center_label: AnyStr,
# torso_label: AnyStr,
# experiment_box_real_length: SupportsFloat,
# center_size_real_length: SupportsFloat,
# length_unit_per_pixel: SupportsFloat,
# max_radians_gaze_and_object: SupportsFloat = 1 / 4 * np.pi,
# exp_id_vs_coordinate_data_path: Dict,
# fps: SupportsFloat,
# coordinate_data_format: AnyStr = "deeplabcut",